# <p align="center"> Identifying and Removing all the Industry data </p>

In [3]:
#Initialize Notebook Environment
from pathlib import Path
import sys
sys.path.insert( 0, Path("../..").resolve().absolute().__str__() )

In [4]:
# Load all session directories
from xaidar.filesUtils import loadPickle

lst_sessions = list( loadPickle("lst_sessions.pkl") )
print(f"Loaded {len(lst_sessions)} sessions from lst_sessions.pkl")
# Example of list of sessions
print(lst_sessions[0] )

Loaded 1007 sessions from lst_sessions.pkl
data/2017/lb18145-17/processing/analysis/TMP_dataset_clustering/lb18145-49


In [5]:
# Filter sessions to only include industry sessions
industry_sessions = []
industry_labels = [ "sw", "in" ]
for session in lst_sessions:
    sesh_label = session.split("/")[-1][:2]
    if sesh_label in industry_labels:
        industry_sessions.append(session)

In [ ]:
# Print examples of industry sessions
print(f"Filtered {len(industry_sessions)} industry sessions from {len(lst_sessions)} total sessions")
# Example of industry sessions
for sesh in industry_sessions[:5]:
    print(sesh)
# print(industry_sessions)

Filtered 65 industry sessions from 1007 total sessions
data/2020/sw24758-9
data/2018/lb19758-9/processing/analysis/TMP_pandda/sw26591-1
data/2021/sw27230-11
data/2017/lb18145-17/processing/analysis/TMP_dataset_clustering/sw24758-9
data/2021/sw29435-7


In [ ]:
# Get the tree objects for the industry sessions
lst_treeObj_sessions = { "tree": [], "path": []}
for sesh in industry_sessions:
    seshPath = sesh.split("/")
    year, session = seshPath[1], seshPath[2]
    lst_treeObj_sessions[ "path" ].append( sesh )
    lst_treeObj_sessions[ "tree" ].append( f"tree_{year}_{session}.pkl")


In [ ]:
# Get the object keys for all objects in industry sessions
from xaidar.treeObj import convertPathtoID , findAllFolderFiles
numb_sessions = len(lst_treeObj_sessions["tree"])
dic_filePaths = {}
print(f"Number of sessions with tree objects: {numb_sessions}")

for idx in range(numb_sessions):
    fileName = lst_treeObj_sessions[ "tree" ][idx]
    dirPath = lst_treeObj_sessions[ "path" ][idx]
    tree = loadPickle( Path(f"../../../data/treeObjs/xchem/data/{ fileName }" ))
    dirID = convertPathtoID( tree["fileTree"], tree["foldersCount"], dirPath)
    # print(f"Session {idx+1}/{numb_sessions}: {dirPath} -> ID: {dirID}")
    results = findAllFolderFiles( tree["fileTree"], tree["foldersCount"], dirID )
    dic_filePaths[ dirPath ] = results

Number of sessions with tree objects: 65


: 

In [8]:
from xaidar.filesUtils import loadPickle
taskDir = Path( "../../../data/tasks/delIndustry" )
numb_files = 0
for file in list( taskDir.glob("[0-9]*.pkl") ):
    lst_paths = loadPickle( file )
    numb_files += len(lst_paths)
    del lst_paths
print(f"Number of files to delete: {numb_files}")

Number of files to delete: 2566835


In [11]:
print(f"Found {len(results)} files in the directory {dirPath} with ID {dirID}")

Found 2905 files in the directory data/2017/lb18145-17/processing/analysis/TMP_dataset_clustering/sw26558-1 with ID [0, 0, 0, 1, 1, 1, 116]


In [12]:
for file in results[:5]:
    print(file)

data/2017/lb18145-17/processing/analysis/TMP_dataset_clustering/sw26558-1/AGIOS1-x0266_normalised.pdb
data/2017/lb18145-17/processing/analysis/TMP_dataset_clustering/sw26558-1/log.txt
data/2017/lb18145-17/processing/analysis/TMP_dataset_clustering/sw26558-1/processed/-1.ccp4
data/2017/lb18145-17/processing/analysis/TMP_dataset_clustering/sw26558-1/processed/0.ccp4
data/2017/lb18145-17/processing/analysis/TMP_dataset_clustering/sw26558-1/processed/1.ccp4


### Delete the Industry Objects

In [ ]:

import os
from pathlib import Path

from xaidar.filesUtils import loadPickle, savePyObj
from xaidar.s3Utils import decryptCredentials, initialize, iterateObjStore, lstAllKeys

# credKey = os.getenv( "CRED_KEY" ) 
credPath =  Path( "../../../credentials.enc").resolve()
credDict = decryptCredentials( credKey, credPath )
client = initialize( "XChem", cred_dict=credDict)

totalSize = 0 
size_failedAPICalls = []


bucket = "xchem"
taskDir = Path( "../../../data/tasks/delIndustry" )
for file in list( taskDir.glob("[0-9]*.pkl") )[:1]:
    lst_paths = loadPickle( file )
    print(lst_paths[:5])


Object Store Names:['XChem', 'MinIO']
Credentials Associated with each Object Store: ['endpoint_url', 'access_key', 'secret_key']
['data/2021/sw27206-8/processing/README.reprocessing']


In [23]:
print(lst_paths[:5])

response = client.head_object(Bucket=bucket, Key="data/2021/sw27206-8/processing/README.reprocessing")

['data/2021/sw27206-8/processing/README.reprocessing']


In [25]:
if not "ContentLength" in list(response.keys()):
    print("ContentLength not found in response keys")
else:
    print(f"ContentLength: {response['ContentLength']} bytes")
    totalSize += response["ContentLength"]

ContentLength: 739 bytes


### Mass Destruction

In [6]:
import os
from pathlib import Path

from xaidar.filesUtils import loadPickle, savePyObj
from xaidar.s3Utils import decryptCredentials, initialize, iterateObjStore, lstAllKeys

# 
# credKey = "" #### !!! Must fill this for code to work !!! ####
# credKey = os.getenv( "CRED_KEY" )
credPath =  Path( "../../../credentials.enc").resolve()
credDict = decryptCredentials( credKey, credPath )
client = initialize( "XChem", cred_dict=credDict)

Object Store Names:['XChem', 'MinIO']
Credentials Associated with each Object Store: ['endpoint_url', 'access_key', 'secret_key']


In [12]:
def massDeletion(client, bucket, lst_paths, window_size=1000, fileName = None):
    numb_paths = len(lst_paths)
    numb_windows = (numb_paths + window_size - 1) // window_size
    for idx in range(0, numb_windows):
        paths_window = lst_paths[idx * window_size : (idx + 1) * window_size]
        paths_input = [  { "Key": path } for path in paths_window ]
        try:
            client.delete_objects(Bucket=bucket, Delete={"Objects": paths_input})
        except Exception as e:
            print(f"Failed to delete objects in window {idx}: {e}")
            continue
    if fileName: print(f"Finished mass deletion of objects for file: {fileName} with {numb_paths} paths")
    else: print(f"Finished mass deletion of objects for {numb_paths} paths")

In [16]:
from xaidar.treeObj import loadPickle
taskDir = Path( "../../../data/tasks/delIndustry" )
filePath = taskDir / "7_sw24987-1_delFiles.pkl"
file = loadPickle( filePath )
print(f"Loaded {len(file)} paths from {filePath}")
bucket = "xchem"
print( file[:2])


Loaded 1771 paths from ../../../data/tasks/delIndustry/7_sw24987-1_delFiles.pkl
['data/2018/lb19758-9/processing/analysis/TMP_pandda/sw24987-1/log.txt', 'data/2018/lb19758-9/processing/analysis/TMP_pandda/sw24987-1/luigi.finished']


In [15]:
massDeletion(client, bucket, file, fileName="7_sw24987-1_delFiles.pkl")

Finished mass deletion of objects for file: 7_sw24987-1_delFiles.pkl with 1771 paths


In [ ]:

bucket = "xchem"
for file in taskDir.glob("[0-9][0-9]*.pkl"):#  # [0-9]*.pkl
    print(f"Processing file: {file.name}, Number of paths: {len(lst_paths)}")
    lst_paths = loadPickle(file)
    massDeletion(client, bucket, lst_paths, fileName=file.name)


for file in taskDir.glob("[789]*.pkl"):#  # [0-9]*.pkl
    print(f"Processing file: {file.name}, Number of paths: {len(lst_paths)}")
    lst_paths = loadPickle(file)
    massDeletion(client, bucket, lst_paths, fileName=file.name)

Processing file: 7_sw24987-1_delFiles.pkl
Processing file: 8_sw26591-1_delFiles.pkl
Processing file: 9_sw26591-1_delFiles.pkl


In [94]:
10//10 
rnge = 4
lst = [1, 2, 3, 4, 5] 
length = len(lst)
for idx in range(0, (length+(rnge-1))//rnge  ):
    print(f"Processing index {idx}: {lst[idx*rnge:( idx+1)*rnge]}")
# This will give you the next two elements in the list starting from idx

Processing index 0: [1, 2, 3, 4]
Processing index 1: [5]


In [41]:
8//2

4

### Test if a sample industry object key still exists

In [11]:
import os
from pathlib import Path

from xaidar.filesUtils import loadPickle, savePyObj
from xaidar.s3Utils import decryptCredentials, initialize, iterateObjStore, lstAllKeys

# credKey = os.getenv( "CRED_KEY" ) 
credPath =  Path( "../../../credentials.enc").resolve()
credDict = decryptCredentials( credKey, credPath )
client = initialize( "XChem", cred_dict=credDict)



Object Store Names:['XChem', 'MinIO']
Credentials Associated with each Object Store: ['endpoint_url', 'access_key', 'secret_key']


In [17]:
bucket = "xchem"
# key = "data/2020/sw26557-1/processing/analysis/model_building/SW26557-x0192/autoprocessing/sw26557-10-SW26557-x0192_2_xia2-dials_c13d8476-b5c6-43df-a304-1f28b0e84f9c/sw26557v10_xSW26557x01922_SAD_merging-statistics.json"
key = "data/2018/lb19758-9/processing/analysis/TMP_pandda/sw24987-1/log.txt"
response = client.head_object(Bucket=bucket, Key=key)
print( response)

ClientError: An error occurred (404) when calling the HeadObject operation: Not Found

In [ ]:
bucket = "xchem"
key = "data/2020/sw26557-1/processing/analysis/model_building/SW26557-x0192/autoprocessing/sw26557-10-SW26557-x0192_2_xia2-dials_c13d8476-b5c6-43df-a304-1f28b0e84f9c/sw26557v10_xSW26557x01922_SAD_merging-statistics.json"

deletedFilesCount = 0
try:
    response = client.head_object(Bucket=bucket, Key=key)
    break
    print(f"Successfully accessed {key} in bucket {bucket}")
except Exception as e:
    print(f"Error accessing {key} in bucket {bucket}: {e}")
    deletedFilesCount += 1


Error accessing data/2020/sw26557-1/processing/analysis/model_building/SW26557-x0192/autoprocessing/sw26557-10-SW26557-x0192_2_xia2-dials_c13d8476-b5c6-43df-a304-1f28b0e84f9c/sw26557v10_xSW26557x01922_SAD_merging-statistics.json in bucket xchem: An error occurred (404) when calling the HeadObject operation: Not Found


In [26]:
def foo():
    for i in range(10):
        for j in range(10):
            try:
                1 / (i%2)
                print( i, j)
                return
            except Exception as e:
                pass
foo()



1 0
